# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, overview, and process the FAIR^2 Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review record sets and their available fields. All references are to be made using their `@id` values as per the Croissant schema.

Let's display the available RecordSets, their Fields, and Columns using their `@id`s.

In [ ]:
# List all RecordSets and their details
record_sets = dataset.record_sets

print("Available Record Sets and their Fields:\n")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")

    # List Fields within RecordSet
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields and Columns:")
    for field in fields:
        print(f"    * Field @id: {field['@id']}")
        print(f"      Name: {field.get('name', 'N/A')}")
        # Some fields may have 'column' keys pointing to columns in files
        col = field.get('column')
        if col:
            col = col['@id'] if isinstance(col, dict) and '@id' in col else col
            print(f"      Column @id: {col}")
    print()

## 3. Data Extraction
Load data from a chosen RecordSet into a DataFrame for analysis.

**Note:** In this notebook, all references are made strictly via `@id` as per the Croissant schema.

First, let's display a few records from the primary RecordSet. You can customize the `record_set_id` variable to any available RecordSet.

In [ ]:
# Select one or more RecordSet @ids for extraction
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display DataFrame columns and preview rows for the first RecordSet
if len(record_set_ids) > 0:
    first_rs = record_set_ids[0]
    print(f"Columns in RecordSet {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    print("\nFirst few records:")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply common data processing steps, including filtering, normalization, and grouping for the dataset loaded from a selected RecordSet.

Pick a numeric field to illustrate filtering and normalization. We'll use Field `@id` values as required (please reference the outputs in the previous section to select an available field).

In [ ]:
# Modify the following IDs according to the actual "Fields" and columns from Overview (Section 2)

# Example selection (please change according to the output above for your analysis):
record_set_id = record_set_ids[0]

# Pick a numeric field - update with your actual Field @id or DataFrame column name
sample_df = dataframes[record_set_id]
candidate_numeric_fields = sample_df.select_dtypes(include='number').columns.tolist()
print(f"Numeric fields: {candidate_numeric_fields}")

# If there is at least one numeric field, perform further analysis
if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]  # choose the first numeric field
    threshold = sample_df[numeric_field_id].mean()  # Use mean as filter threshold

    filtered_df = sample_df[sample_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (mean):\n")
    display(filtered_df.head())

    # Normalize
    filtered_df[numeric_field_id+'_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id+'_normalized']].head())

    # Try grouping by another field (if available)
    other_fields = [c for c in filtered_df.columns if c != numeric_field_id]
    group_field = other_fields[0] if other_fields else None
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("No numeric fields found to perform EDA.")

## 5. Visualization
Visualize distributions or relationships of the selected fields. Update field references as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Plot only if we have appropriate data
if candidate_numeric_fields:
    # Histogram of chosen numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(sample_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet {record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Boxplot by group_field (if set)
    if group_field is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field} (filtered)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated loading, exploring, and basic processing of a biomedical Croissant dataset using `mlcroissant`.

- RecordSets, fields, and columns can be systematically referenced by their `@id` as required for reproducible FAIR analysis.
- The FAIR² dataset provides protein and peptide-level information extractable and visualizable in a DataFrame-centric workflow.
- For extended analyses, review all available RecordSets and explore other numeric or categorical fields for domain-specific questions.